# `contour(overlay_sets=...)` — Overlaying points & boundaries on a contour

`UnichartNotebook.contour()` interpolates a scattered `(x, y, z)` field into a
filled contour. The **`overlay_sets`** argument lets you draw *other* datasets
on top of that field — using the usual selector (`int` / list / `'all'` /
`Dataset`).

Each overlay set keeps its **full plot style**:

- a set with **no linestyle** shows as **markers** — handy for marking the
  actual sampled / measured operating points over the interpolated field;
- a set **with a linestyle** is drawn as a **connected line** — so it can trace
  a **boundary** (a surge line, an operating limit, an envelope) across the map.

Points are connected in the set's own order (its `order` column if set, else
row order), so a hand-drawn boundary keeps its shape.

We'll use a small **compressor map** as the running example:
`x = mass flow`, `y = pressure ratio`, `z = efficiency`.

## Set up

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

rng = np.random.default_rng(0)

# --- The efficiency FIELD: scattered (flow, PR, eff) samples to be contoured ---
# Efficiency is a peaked "island": highest near (flow=5, PR=2.6), falling off
# quadratically away from that peak.
flow = rng.uniform(2.0, 8.0, 220)
pr   = rng.uniform(1.5, 4.0, 220)
eff  = 0.88 - 0.012 * (flow - 5.0) ** 2 - 0.06 * (pr - 2.6) ** 2
# A second quantity over the same x/y, used later to show two z-subplots.
margin = 4.0 - pr
field = pd.DataFrame({'FLOW': flow, 'PR': pr, 'EFF': eff, 'MARGIN': margin})

# --- A SURGE LINE boundary: a curve along the left edge of the operable region.
# Ordered along the curve so it draws as a clean line.
sx = np.linspace(2.2, 7.5, 12)
surge = pd.DataFrame({'FLOW': sx, 'PR': 1.6 + 0.42 * sx})

# --- Measured OPERATING POINTS: a handful of discrete (flow, PR) test points. ---
ops = pd.DataFrame({'FLOW': [3.5, 4.5, 5.0, 5.5, 6.5],
                    'PR':   [2.0, 2.4, 2.7, 3.0, 3.4]})

uc = UnichartNotebook()
uc.load(field, title='Efficiency field')   # -> Set 0
uc.load(surge, title='Surge line')          # -> Set 1
uc.load(ops,   title='Operating points')    # -> Set 2

UniChart Notebook Environment Initialized.
Loaded Set 0: Efficiency field
Loaded Set 1: Surge line
Loaded Set 2: Operating points


## 1. The plain contour (no overlay)

First, just the interpolated efficiency field. Only Set 0 is selected, because
`contour()` builds its surface from the **selected** datasets.

In [2]:
uc.select(0)
uc.contour(x='FLOW', y='PR', z='EFF', suptitle='Compressor efficiency map')

## 2. Overlay the measured operating points (markers)

The operating-points set (Set 2) has no linestyle, so it overlays as **markers**.
We give it a visible style first. Note `overlay_sets` is independent of the
*selected* sets — Set 2 is overlaid even though only Set 0 builds the field.

In [3]:
uc.markersize(2, 14)
uc.marker(2, 'o')
uc.color(2, 'black')
uc.linestyle(2, "-")
uc.select(0)
uc.contour(x='FLOW', y='PR', z='EFF', overlay_sets=2,
           suptitle='Efficiency map + measured operating points')

## 3. Overlay a boundary line (surge line)

Give the surge-line set (Set 1) a **linestyle** — now it draws as a connected
line tracing the boundary, instead of markers. We overlay *both* the boundary
and the operating points by passing a list.

In [4]:
uc.linestyle(1, '--')
uc.linewidth(1, 3)
uc.color(1, 'black')
uc.marker(1, None)          # line only, no marker glyphs

uc.select(0)
uc.contour(x='FLOW', y='PR', z='EFF', overlay_sets=[1, 2],
           suptitle='Efficiency map + surge line + operating points')

## 4. Hollow markers via `fill`

Turning `fill` off makes the markers hollow — their outline takes the set
color. Useful when you don't want overlay markers to hide the field beneath.

In [5]:
uc.fill(2, False)
uc.markersize(2, 16)

uc.select(0)
fig = uc.contour(x='FLOW', y='PR', z='EFF', overlay_sets=[1, 2],
                 suptitle='Hollow operating-point markers')

uc.fill(2, True)            # restore for later cells
fig                         # (display the figure built above)

## 5. Overlays repeat on every subplot

With `by='sets'` each subplot is a *different* contour dataset, and with
multiple `z` variables (`by='vars'`) each subplot is a different `z`. In both
cases the **same overlay sets are drawn on every subplot**, since all contour
subplots share the same x/y axes.

Here we add a second z-variable (a pressure-ratio margin) and plot both — the
surge line and operating points appear on each panel.

In [6]:
# Set 0 already carries a second column, MARGIN, over the same x/y — so asking
# for two z-variables gives two subplots, and the overlays repeat on each.
uc.select(0)
uc.contour(x='FLOW', y='PR', z=['EFF', 'MARGIN'], overlay_sets=[1, 2], ncols=2,
           suptitle='Two fields, same overlays on each')

## Recap

- `overlay_sets` accepts the usual dataset selector and is **independent** of
  which sets build the contour.
- **No linestyle → markers** (sample points); **linestyle → line** (boundary).
- Style is fully honored: `color`, `marker`, `markersize`, `linewidth`,
  `linestyle`, `fill`.
- Overlays are drawn on **every** subplot and always **on top** of the fill.